# String Prompt Templates

The `string.py` module provides utilities for formatting, validating, inspecting, and representing string-based prompt templates. It supports `f-string`, Mustache, and Jinja2 template formats.

# PromptTemplateFormat: `TypeAlias`

`PromptTemplateFormat` represents the supported string-template formats.

**Syntax**

```python
PromptTemplateFormat = Literal["f-string", "mustache", "jinja2"]
```

# Defs: `TypeAlias`

`Defs` represents a recursive dictionary structure used to build nested Pydantic models for Mustache templates.

**Syntax**

```python
Defs = dict[str, "Defs"]
```

# Constants

1. `DEFAULT_FORMATTER_MAPPING`:`dict[str, Callable[..., str]]`:= Maps each supported template format to its formatting function.

2. `DEFAULT_VALIDATOR_MAPPING`:`dict[str, Callable[[str, list[str]], None]]`:= Maps supported template formats to their validation functions.

# Functions

1. `jinja2_formatter`:= Formats a template using Jinja2's sandboxed environment.

   Jinja2 sandboxing is a best-effort security measure and should not be used with untrusted or user-controlled templates.

   **Syntax**

   ```python
   jinja2_formatter(
       template: str, # Jinja2 template string
       /,
       **kwargs: Any # Variables used to format the template
   ) -> str
   ```

2. `validate_jinja2`:= Checks whether the declared input variables match the variables used in a Jinja2 template.

   It issues a warning when variables are missing or unused variables are supplied.

   **Syntax**

   ```python
   validate_jinja2(
       template: str, # Jinja2 template string
       input_variables: list[str] # Expected input-variable names
   ) -> None
   ```

3. `mustache_formatter`:= Formats a template using Mustache.

   **Syntax**

   ```python
   mustache_formatter(
       template: str, # Mustache template string
       /,
       **kwargs: Any # Variables used to format the template
   ) -> str
   ```

4. `mustache_template_vars`:= Returns the top-level variables used in a Mustache template.

   For nested variables such as `person.name`, only the top-level key `person` is returned.

   **Syntax**

   ```python
   mustache_template_vars(
       template: str # Mustache template string
   ) -> set[str]
   ```

5. `mustache_schema`:= Creates a Pydantic input model from the variables and nested sections in a Mustache template.

   **Syntax**

   ```python
   mustache_schema(
       template: str # Mustache template string
   ) -> type[BaseModel]
   ```

6. `validate_f_string_template`:= Validates an f-string template and returns its sorted input-variable names.

   It rejects attribute access, indexing, numeric-only variable names, and nested replacement fields inside format specifiers.

   **Syntax**

   ```python
   validate_f_string_template(
       template: str # F-string template to validate
   ) -> list[str]
   ```

7. `check_valid_template`:= Validates a template against its format and expected input variables.

   It raises an error when the template format is unsupported or the prompt schema is invalid.

   **Syntax**

   ```python
   check_valid_template(
       template: str, # Template string to validate
       template_format: str, # Template format
       input_variables: list[str] # Expected input-variable names
   ) -> None
   ```

8. `get_template_variables`:= Extracts and returns the sorted variables used in a template.

   **Syntax**

   ```python
   get_template_variables(
       template: str, # Template string to inspect
       template_format: str # Template format
   ) -> list[str]
   ```

9. `is_subsequence`:= Checks whether one sequence is a prefix-aligned subsequence of another sequence.

   **Syntax**

   ```python
   is_subsequence(
       child: Sequence[Any], # Sequence to compare
       parent: Sequence[Any] # Sequence that may contain the child
   ) -> bool
   ```

# StringPromptTemplate: `BasePromptTemplate[str]`, `ABC`

`StringPromptTemplate` is an abstract base class for prompt templates that produce string-based prompt values.

This abstract class is not intended to be instantiated directly.

## Methods

1. `get_lc_namespace`:= Returns the LangChain serialization namespace assigned to the class.

   **Syntax**

   ```python
   @classmethod
   get_lc_namespace(
       cls # String prompt template class
   ) -> list[str]
   ```

2. `format_prompt`:= Synchronously formats the prompt and wraps the resulting text in a `StringPromptValue`.

   **Syntax**

   ```python
   format_prompt(
       self, # String prompt template instance
       **kwargs: Any # Values used to format the prompt
   ) -> PromptValue
   ```

3. `aformat_prompt`:= Asynchronously formats the prompt and wraps the resulting text in a `StringPromptValue`.

   **Syntax**

   ```python
   async aformat_prompt(
       self, # String prompt template instance
       **kwargs: Any # Values used to format the prompt
   ) -> PromptValue
   ```

4. `format`:= Defines how the prompt template is synchronously formatted into a string.

   Subclasses must implement this abstract method.

   **Syntax**

   ```python
   @abstractmethod
   format(
       self, # String prompt template instance
       **kwargs: Any # Values used to format the prompt
   ) -> str
   ```

5. `pretty_repr`:= Returns a human-readable representation of the prompt template.

   When `html` is enabled, input-variable placeholders are colour-formatted.

   **Syntax**

   ```python
   pretty_repr(
       self, # String prompt template instance
       html: bool = False # Whether to return an HTML-formatted representation
   ) -> str
   ```

6. `pretty_print`:= Prints a human-readable representation of the prompt template.

   Interactive environments use the HTML-style representation.

   **Syntax**

   ```python
   pretty_print(
       self # String prompt template instance
   ) -> None
   ```


In [ ]:
from langchain_core.prompts import PromptTemplate # Import a concrete string prompt template
from langchain_core.prompts.string import check_valid_template, get_template_variables, is_subsequence, jinja2_formatter, mustache_formatter, mustache_schema, mustache_template_vars # Import string-template utilities

f_string_template = "Hello, {name} from {city}!" # Define an f-string template
variables = get_template_variables(f_string_template, "f-string") # Extract the template variables
print(variables) # Display the sorted variable names

check_valid_template(f_string_template, "f-string", ["name", "city"]) # Validate the template and expected variables

mustache_template = "Hello, {{person.name}}!" # Define a Mustache template with a nested variable
print(mustache_template_vars(mustache_template)) # Display the top-level Mustache variable
print(mustache_formatter(mustache_template, person={"name": "Saad"})) # Format the Mustache template

input_model = mustache_schema(mustache_template) # Generate a Pydantic model from the Mustache structure
print(input_model.model_json_schema()) # Display the generated input schema

jinja2_template = "Hello, {{ name }}!" # Define a Jinja2 template
print(jinja2_formatter(jinja2_template, name="Saad")) # Format the Jinja2 template

print(is_subsequence(["name"], ["name", "city"])) # Check whether the child sequence matches the beginning of the parent

prompt = PromptTemplate.from_template("Welcome, {name}!") # Create a concrete StringPromptTemplate subclass
prompt_value = prompt.format_prompt(name="Saad") # Format the prompt synchronously as a StringPromptValue
print(prompt_value.to_string()) # Display the synchronous prompt text

async_prompt_value = await prompt.aformat_prompt(name="Arfin") # Format the prompt asynchronously in Jupyter
print(async_prompt_value.to_string()) # Display the asynchronous prompt text

print(prompt.pretty_repr()) # Display a readable representation of the prompt
prompt.pretty_print() # Print the readable representation